In [44]:
import pandas as pd
# Cell 2: Define BSCLoader class with separate loading
class BSCLoader:
    """Load and process BSC data from Excel"""
    
    def __init__(self, file_path):
        self.file_path = Path(file_path)
        self.rbb_raw = None
        self.digital_raw = None
        self.rbb_clean = None
        self.digital_clean = None
        self.combined_data = None
        
    def load_separately(self):
        """Load RBB and Digital sheets separately"""
        print(f"\n📂 Loading file: {self.file_path.name}")
        
        # Load RBB sheet
        try:
            self.rbb_raw = pd.read_excel(self.file_path, sheet_name=' RBB 2025.2026', header=None)
            print(f"  ✓ Loaded RBB 2025.2026: {len(self.rbb_raw)} rows, {len(self.rbb_raw.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading RBB sheet: {e}")
        
        # Load Digital sheet
        try:
            self.digital_raw = pd.read_excel(self.file_path, sheet_name='Digital 2025.2026', header=None)
            print(f"  ✓ Loaded Digital 2025.2026: {len(self.digital_raw)} rows, {len(self.digital_raw.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading Digital sheet: {e}")
        
        return self.rbb_raw, self.digital_raw
    
    def clean_rbb(self):
        """Clean and extract RBB data"""
        if self.rbb_raw is None:
            print("❌ RBB data not loaded yet")
            return pd.DataFrame()
        
        print("\n📋 Cleaning RBB data...")
        data_rows = []
        
        # Data starts from row 2 (index 2)
        for idx in range(2, len(self.rbb_raw)):
            row = self.rbb_raw.iloc[idx]
            
            # Get Strategic Objective from column A (index 0)
            strategic_obj = row.iloc[0] if len(row) > 0 else None
            
            # Stop when we hit empty or TOTAL
            if pd.isna(strategic_obj) or str(strategic_obj).strip() == '':
                if idx > 20:  # Break after row 20 if empty
                    break
                continue
            
            if 'TOTAL' in str(strategic_obj).upper():
                break
            
            # Get KPI from column B (index 1)
            kpi = row.iloc[1] if len(row) > 1 else None
            
            # Skip if no KPI
            if pd.isna(kpi) or str(kpi).strip() == '':
                continue
            
            # Get other columns
            measurement = row.iloc[2] if len(row) > 2 else None
            weight = row.iloc[3] if len(row) > 3 else None
            plan = row.iloc[9] if len(row) > 9 else None
            
            # Clean weight
            if weight and isinstance(weight, (int, float)):
                weight = float(weight)
            elif weight and isinstance(weight, str):
                weight = float(re.sub(r'[^0-9.]', '', weight)) if re.search(r'\d', weight) else None
            
            # Clean measurement
            measurement = str(measurement).strip() if pd.notna(measurement) else ''
            
            data_rows.append({
                'division': 'RBB',
                'strategic_objective': str(strategic_obj).strip(),
                'kpi': str(kpi).strip(),
                'measurement': measurement,
                'weight': weight if weight else 0,
                'plan': plan if pd.notna(plan) else None
            })
        
        self.rbb_clean = pd.DataFrame(data_rows)
        print(f"  ✅ Cleaned {len(self.rbb_clean)} RBB KPIs")
        return self.rbb_clean
    
    def clean_digital(self):
        """Clean and extract Digital data"""
        if self.digital_raw is None:
            print("❌ Digital data not loaded yet")
            return pd.DataFrame()
        
        print("\n📋 Cleaning Digital data...")
        data_rows = []
        current_strategic_obj = None
        
        # Data starts from row 2 (index 2)
        for idx in range(2, len(self.digital_raw)):
            row = self.digital_raw.iloc[idx]
            
            # Get Strategic Objective from column A (index 0)
            strategic_obj = row.iloc[0] if len(row) > 0 else None
            
            # Handle merged cells - carry forward last strategic objective
            if pd.notna(strategic_obj) and str(strategic_obj).strip() != '':
                if 'TOTAL' not in str(strategic_obj).upper():
                    current_strategic_obj = str(strategic_obj).strip()
            
            # Stop at TOTAL
            if current_strategic_obj and 'TOTAL' in current_strategic_obj.upper():
                break
            
            # Get KPI from column B (index 1)
            kpi = row.iloc[1] if len(row) > 1 else None
            
            # Skip if no KPI
            if pd.isna(kpi) or str(kpi).strip() == '':
                continue
            
            # Skip header rows
            if 'Major - KPI' in str(kpi):
                continue
            
            # Get other columns
            measurement = row.iloc[2] if len(row) > 2 else None
            weight = row.iloc[3] if len(row) > 3 else None
            plan = row.iloc[9] if len(row) > 9 else None
            
            # Clean weight
            if weight and isinstance(weight, (int, float)):
                weight = float(weight)
            elif weight and isinstance(weight, str):
                weight = float(re.sub(r'[^0-9.]', '', weight)) if re.search(r'\d', weight) else None
            
            # Clean measurement
            measurement = str(measurement).strip() if pd.notna(measurement) else ''
            
            if current_strategic_obj:
                data_rows.append({
                    'division': 'Digital',
                    'strategic_objective': current_strategic_obj,
                    'kpi': str(kpi).strip(),
                    'measurement': measurement,
                    'weight': weight if weight else 0,
                    'plan': plan if pd.notna(plan) else None
                })
        
        self.digital_clean = pd.DataFrame(data_rows)
        print(f"  ✅ Cleaned {len(self.digital_clean)} Digital KPIs")
        return self.digital_clean
    
    def merge_data(self):
        """Merge RBB and Digital data into one dataset"""
        if self.rbb_clean is None:
            self.clean_rbb()
        if self.digital_clean is None:
            self.clean_digital()
        
        print("\n🔗 Merging RBB and Digital data...")
        self.combined_data = pd.concat([self.rbb_clean, self.digital_clean], ignore_index=True)
        print(f"  ✅ Merged {len(self.combined_data)} total KPIs")
        print(f"     - RBB: {len(self.rbb_clean)}")
        print(f"     - Digital: {len(self.digital_clean)}")
        
        return self.combined_data

In [45]:
# Cell 3: Initialize loader and load data separately
import os

file_path = "./Data/raw/Bsc_of_DB_and_RBB_divisions.xlsx"

# Check if file exists
if os.path.exists(file_path):
    print(f"✅ File found at: {file_path}")
else:
    print(f"❌ File NOT found at: {file_path}")
    # Try to find it
    for root, dirs, files in os.walk("."):
        for file in files:
            if "Bsc" in file and ".xlsx" in file:
                print(f"   Found: {os.path.join(root, file)}")
                file_path = os.path.join(root, file)
                break

loader = BSCLoader(file_path)

# Load sheets separately
rbb_raw, digital_raw = loader.load_separately()

✅ File found at: ./Data/raw/Bsc_of_DB_and_RBB_divisions.xlsx

📂 Loading file: Bsc_of_DB_and_RBB_divisions.xlsx
  ✓ Loaded RBB 2025.2026: 32 rows, 15 columns
  ✓ Loaded Digital 2025.2026: 51 rows, 15 columns


In [46]:
# Cell 4: Clean each dataset separately
print("\n" + "="*80)
print("CLEANING DATA SEPARATELY")
print("="*80)

# Clean RBB
rbb_clean = loader.clean_rbb()
print(f"\n📋 RBB Clean Data Preview:")
display(rbb_clean.head(10))

# Clean Digital
digital_clean = loader.clean_digital()
print(f"\n📋 Digital Clean Data Preview:")
display(digital_clean.head(10))


CLEANING DATA SEPARATELY

📋 Cleaning RBB data...
  ✅ Cleaned 11 RBB KPIs

📋 RBB Clean Data Preview:


,division,strategic_objective,kpi,measurement,weight,plan
0,RBB,Strategic Objectives,Major - KPI,Measurement,0.0,Plan
1,RBB,Enhance Deposit Mobilization,Deposit from Retail Customers,Million birr,25.0,456610.7
2,RBB,Enhance Customer Value,Mobilize 0.3% of the annual deposit Plan from ...,Million birr,4.0,1369.8
3,RBB,Increase FCY Earning,Remittance,Thousand USD,10.0,1313357
4,RBB,Ensure Prudent Resource Allocation,Loan Disbursement,Million birr,6.0,95581.3
5,RBB,Ensure Financial Soundness,Loan Collection,Million birr,6.0,24761.8
6,RBB,Increase income,Non interest income,Million birr,10.0,15792.7
7,RBB,Manage Operational Expense,Operational Expense,Million birr,10.0,4827.7
8,RBB,Increase Customer Base,New Retail Deposit Customer recruitment,Number,2.0,5019371
9,RBB,Enhance Customer Experience,Customer satisfaction rate,%,2.0,>80%



📋 Cleaning Digital data...
  ✅ Cleaned 38 Digital KPIs

📋 Digital Clean Data Preview:


,division,strategic_objective,kpi,measurement,weight,plan
0,Digital,Enhance resource mobilization from Digital cha...,Amount Mobilization from Acquiring Internation...,Million birr,5.0,1337.3
1,Digital,Enhance resource mobilization from Digital cha...,Amount FCY mobilized through relationships wit...,Thousand USD,9.0,75564.1
2,Digital,Increase income from digital channels,Income From Digital Banking,Million birr,13.0,10256.7
3,Digital,Enhance digital channel utilization,New Active card banking users recruitment,Number,1.0,3578733
4,Digital,Enhance digital channel utilization,Active card banking users Incremental,Number,1.0,5511447
5,Digital,Enhance digital channel utilization,Active card banking users,%,2.0,0.329
6,Digital,Enhance digital channel utilization,New Active mobile banking users recruitment,Number,1.0,6539114
7,Digital,Enhance digital channel utilization,Active mobile banking users Incremental,Number,1.0,8143652
8,Digital,Enhance digital channel utilization,Active mobile banking users,%,2.0,0.678
9,Digital,Enhance digital channel utilization,New Active internet banking users recruitment,Number,1.0,679994


In [47]:
# Cell 5: Merge the two data sources
print("\n" + "="*80)
print("MERGING RBB AND DIGITAL DATA")
print("="*80)

combined_data = loader.merge_data()
print(f"\n📊 Combined Data Shape: {combined_data.shape}")
print(f"\n📋 Combined Data Preview:")
display(combined_data.head(15))


MERGING RBB AND DIGITAL DATA

🔗 Merging RBB and Digital data...
  ✅ Merged 49 total KPIs
     - RBB: 11
     - Digital: 38

📊 Combined Data Shape: (49, 6)

📋 Combined Data Preview:


,division,strategic_objective,kpi,measurement,weight,plan
0,RBB,Strategic Objectives,Major - KPI,Measurement,0.0,Plan
1,RBB,Enhance Deposit Mobilization,Deposit from Retail Customers,Million birr,25.0,456610.7
2,RBB,Enhance Customer Value,Mobilize 0.3% of the annual deposit Plan from ...,Million birr,4.0,1369.8
3,RBB,Increase FCY Earning,Remittance,Thousand USD,10.0,1313357
4,RBB,Ensure Prudent Resource Allocation,Loan Disbursement,Million birr,6.0,95581.3
5,RBB,Ensure Financial Soundness,Loan Collection,Million birr,6.0,24761.8
6,RBB,Increase income,Non interest income,Million birr,10.0,15792.7
7,RBB,Manage Operational Expense,Operational Expense,Million birr,10.0,4827.7
8,RBB,Increase Customer Base,New Retail Deposit Customer recruitment,Number,2.0,5019371
9,RBB,Enhance Customer Experience,Customer satisfaction rate,%,2.0,>80%


In [ ]:
# Cell 1: Import and define extraction function
import pandas as pd
from docx import Document
import re

def extract_responsibilities(text):
    """Extract bullet responsibilities from JD text."""
    pattern = r"key job duties and responsibilities:(.*?)(iv\.|v\.|vi\.|$)"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    
    if not match:
        return []
    
    section = match.group(1)
    bullets = re.split(r"[•\-]\s*", section)
    
    responsibilities = [
        b.strip() for b in bullets
        if len(b.strip()) > 20
    ]
    
    return responsibilities


def extract_all_jd_tables(file_path):
    """Extract all job descriptions from Word document tables."""
    doc = Document(file_path)
    jd_records = []
    
    for table in doc.tables:
        table_text = ""
        
        for row in table.rows:
            row_text = " | ".join([cell.text.strip() for cell in row.cells if cell.text.strip()])
            table_text += row_text + "\n"
        
        table_text = table_text.lower()
        table_text = re.sub(r'\s+', ' ', table_text)
        
        def extract(field):
            pattern = rf"{field}\s*:\s*([^|]+)"
            match = re.search(pattern, table_text)
            return match.group(1).strip() if match else None
        
        responsibilities = extract_responsibilities(table_text)
        
        jd_records.append({
            "job_title": extract("job title"),
            "division": extract("division"),
            "department": extract("department"),
            "job_grade": extract("job grade"),
            "job_category": extract("job category"),
            "job_objective": extract("job objective"),
            "responsibilities": responsibilities,
        })
    
    df = pd.DataFrame(jd_records)
    return df

print("✅ Functions defined")

✅ Functions defined


In [49]:
# Cell 2: Extract raw data
file_path = "./Data/raw/JD2.docx"  # Update your path

raw_jd_df = extract_all_jd_tables(file_path)

print(f"✅ Extracted {len(raw_jd_df)} job descriptions")

✅ Extracted 23 job descriptions


In [50]:
# Cell 3: View the raw data
print("RAW JD DATA:")
print("="*80)
display(raw_jd_df)

RAW JD DATA:


,job_title,division,department,job_grade,job_category,job_objective,responsibilities,full_text
0,branch manager special branch,retail & branch banking,district,15 unit: branch,middle level management: mlm,to ensure the overall banking operation activi...,"[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
1,branch business manager,retail & branch banking,district,13 unit: branch,operative level management: olm,to ensure the effectiveness of the branch sale...,"[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
2,customer service manager -sales,retail & branch banking,district,12 unit: branch,operative level management: olm,"to sale bank products and services, mobilizes ...",[the job holder’s responsibilities include but...,job details /profile/: | job details /profile/...
3,customer service manager -service,retail & branch banking,district,12 unit: branch,operative level management: olm,"to serve the bank’s customers, mobilizes resou...","[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
4,senior branch banking officer-sales,retail & branch banking,district,11 unit: branch,experienced professional: ep,"to sale bank products and services, mobilizes ...","[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
5,senior branch banking officer-service,retail & branch banking,district,11 unit: branch,experienced professional: ep,"to serve the bank’s customers, mobilizes resou...","[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
6,branch banking officer,retail & branch banking,district,9 unit: branch,professional: p,"to sale bank products and services, expands br...",[],job details /profile/: | job details /profile/...
7,branch banking officer-service,retail & branch banking,district,9 unit: branch,professional: p,"to serve the bank’s customers, mobilizes resou...","[the job holder’s responsibilities include, bu...",job details /profile/: | job details /profile/...
8,retail customer relationship manager,retail & branch banking,district,12 unit: branch,experienced professional: ep,to attend affluent retail customers request at...,"[the job holder’s responsibilities include, bu...",job details /profile/: job title: retail custo...
9,wholesale customer relationship manager,retail & branch banking,district,12 unit: branch,experienced professional: ep,to attend affluent retail customers request at...,"[the job holder’s responsibilities include, bu...",job details /profile/: job title: wholesale cu...
